# Amazon Fulfillment Center Employee Voice Analysis

For this project, I wanted to find out what employees say about working in Amazon fulfillment centers and which issues show up most often in negative feedback. I used Python and NLP to organize reviews, classify sentiment, identify workplace topics, and compare the findings across employee groups.

I originally explored training as a possible reason for turnover, but after reviewing the feedback, workload and scheduling stood out more. This notebook shows how I got those results.

**My dataset:** 138 Glassdoor reviews and 10 YouTube video transcripts. These are selected public comments, not a representative survey of Amazon employees. I don't have Amazon's actual turnover records, so this project looks at potential workplace concerns rather than proven reasons employees leave.

**Before running:** Start from the repository's main folder. You'll need authorized copies of the two source CSVs in `data/source/`. Their filenames include `_cleaned`, but I used the original review and transcript text fields rather than older sentiment predictions or aggressively filtered text. I kept original feedback and derived labels separate.

This notebook is the organized version of my Colab analysis. I left out earlier experiments with different models and passage-splitting methods so the main workflow is easier to follow.

## 1. Get the feedback ready

I started by loading the Glassdoor and YouTube files and choosing the information useful for my analysis. I kept Glassdoor's pros and cons separate because one person might like the benefits but dislike their workload. I also checked missing values and duplicate records, then standardized the column names and dates without changing what people wrote.

In [ ]:
import pandas as pd
import os
import re
import html
from collections import Counter

os.makedirs("data/processed", exist_ok=True)
glassdoor = pd.read_csv("data/source/glassdoor_cleaned.csv")
youtube = pd.read_csv("data/source/youtube_dataset_cleaned.csv")
print("Source shapes:", glassdoor.shape, youtube.shape)

In [ ]:
# step 2: select original review info

# Glassdoor
gd = glassdoor[[
    "review_id",
    "rating_date",
    "rating_overall",
    "employee_status",
    "employee_job_title",
    "review_pros",
    "review_cons"
]].copy()

gd["source"] = "Glassdoor"

# YouTube
yt = youtube[[
    "Link",
    "Upload Date",
    "Transcript"
]].copy()

yt = yt.rename(columns={
    "Link": "source_url",
    "Upload Date": "review_date",
    "Transcript": "review_text"
})

yt["source"] = "YouTube"

# Preview both datasets
display(gd.head())
display(yt.head())

In [ ]:
# step 3: check data quality

print("GLASSDOOR MISSING VALUES:")
print(gd.isnull().sum())

print("\nYOUTUBE MISSING VALUES:")
print(yt.isnull().sum())

print("\nGLASSDOOR DUPLICATES:")
print(gd.duplicated(subset=["review_id"]).sum())

print("\nYOUTUBE DUPLICATES:")
print(yt.duplicated(subset=["source_url"]).sum())

In [ ]:
# step 4: handle missing employee information

gd["employee_status"] = gd["employee_status"].fillna("Unknown")

gd["employee_job_title"] = gd["employee_job_title"].fillna("Unknown")

# Verify missing values are handled
print("GLASSDOOR MISSING VALUES:")
print(gd.isnull().sum())

print("\nGLASSDOOR EMPLOYEE STATUS:")
print(gd["employee_status"].value_counts())

print("\nDATASET SIZES:")
print("Glassdoor:", gd.shape)
print("YouTube:", yt.shape)

In [ ]:
# step 5: standardize dates

# Convert Glassdoor dates
gd["rating_date"] = pd.to_datetime(
    gd["rating_date"], errors="coerce", utc=True
)

# Convert YouTube dates
yt["review_date"] = pd.to_datetime(
    yt["review_date"], errors="coerce", utc=True
)

# Check for dates that failed conversion
print("GLASSDOOR INVALID DATES:",
      gd["rating_date"].isna().sum())

print("YOUTUBE INVALID DATES:",
      yt["review_date"].isna().sum())

# Preview converted dates
display(gd[["rating_date"]].head())
display(yt[["review_date"]].head())

In [ ]:
# step 6: standardize column names

# Glassdoor
gd = gd.rename(columns={
    "rating_date": "review_date",
    "rating_overall": "rating"
})

gd["source_url"] = pd.NA
gd["review_text"] = pd.NA

# YouTube
yt["review_id"] = [
    f"YT_{i+1}" for i in range(len(yt))
]

yt["rating"] = pd.NA
yt["employee_status"] = pd.NA
yt["employee_job_title"] = pd.NA
yt["review_pros"] = pd.NA
yt["review_cons"] = pd.NA

# Use the same columns in both datasets
columns = [
    "review_id",
    "source",
    "source_url",
    "review_date",
    "rating",
    "employee_status",
    "employee_job_title",
    "review_pros",
    "review_cons",
    "review_text"
]

gd = gd[columns]
yt = yt[columns]

# Check results
display(gd.head(2))
display(yt.head(2))

In [ ]:
# step 7: Combine Glassdoor and YouTube data

# Combine both datasets
combined = pd.concat([gd, yt], ignore_index=True)

# Create a unique ID for each record
combined["record_id"] = (
    combined["source"].str.upper()
    + "_"
    + combined["review_id"].astype(str)
)

# Check the combined dataset
print("TOTAL RECORDS:", len(combined))

print("\nRECORDS BY SOURCE:")
print(combined["source"].value_counts())

print("\nDUPLICATE RECORD IDS:")
print(combined["record_id"].duplicated().sum())

print("\nFINAL DATASET SHAPE:")
print(combined.shape)

display(combined.head())

In [ ]:
# step 8: Prepare review text

import html
import re

def prepare_text(text):
    if pd.isna(text):
        return pd.NA

    # Convert HTML entities such as &amp; into &
    text = html.unescape(str(text))

    # Replace line breaks and extra spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Prepare each text field separately
for col in ["review_pros", "review_cons", "review_text"]:
    combined[col + "_prepared"] = (
        combined[col].apply(prepare_text)
    )

# Preview original vs prepared text
display(combined[[
    "source",
    "review_cons",
    "review_cons_prepared"
]].head(5))

print("DATASET SHAPE:", combined.shape)

In [ ]:
# step 9: create an analysis-ready feedback dataset

feedback_rows = []

for _, row in combined.iterrows():

    if row["source"] == "Glassdoor":
        fields = [
            ("pros", row["review_pros_prepared"]),
            ("cons", row["review_cons_prepared"])
        ]
    else:
        fields = [
            ("transcript", row["review_text_prepared"])
        ]

    for feedback_type, text in fields:
        if pd.notna(text) and str(text).strip():
            feedback_rows.append({
                "record_id": row["record_id"],
                "source": row["source"],
                "review_date": row["review_date"],
                "rating": row["rating"],
                "employee_status": row["employee_status"],
                "employee_job_title": row["employee_job_title"],
                "feedback_type": feedback_type,
                "feedback_text": text
            })

feedback = pd.DataFrame(feedback_rows)

# give each feedback entry a unique ID
feedback["feedback_id"] = [
    f"FB_{i+1}" for i in range(len(feedback))
]

print("TOTAL FEEDBACK ENTRIES:", len(feedback))

print("\nFEEDBACK BY TYPE:")
print(feedback["feedback_type"].value_counts())

display(feedback.head())

## 2. Break the YouTube transcripts into smaller sections

The YouTube transcripts are much longer than Glassdoor reviews. I split them into overlapping passages so the sentiment model would have a manageable amount of text at a time. I kept each passage linked to its original video. Later, I count each theme once per source record so longer videos don't get extra weight.

In [ ]:
# step 12: rebuild YouTube passages

def split_by_words(text, max_words=80, overlap=15):
    words = str(text).split()
    passages = []
    step = max_words - overlap

    for start in range(0, len(words), step):
        chunk = words[start:start + max_words]

        if chunk:
            passages.append(" ".join(chunk))

        if start + max_words >= len(words):
            break

    return passages

# Start from the original 10 transcript entries
youtube_original = feedback[
    feedback["source"] == "YouTube"
].copy()

youtube_original["passage_text"] = (
    youtube_original["feedback_text"]
    .apply(split_by_words)
)

youtube_passages = youtube_original.explode(
    "passage_text"
).copy()

youtube_passages["feedback_text"] = (
    youtube_passages["passage_text"]
)

youtube_passages = youtube_passages.drop(
    columns=["passage_text"]
)

# Keep the original Glassdoor entries
glassdoor_entries = feedback[
    feedback["source"] == "Glassdoor"
].copy()

# Combine everything again
analysis_data = pd.concat(
    [glassdoor_entries, youtube_passages],
    ignore_index=True
)

# Create unique analysis IDs
analysis_data["analysis_id"] = [
    f"A_{i+1}" for i in range(len(analysis_data))
]

# Check word counts again
analysis_data["word_count"] = (
    analysis_data["feedback_text"]
    .str.split()
    .str.len()
)

print("TOTAL ANALYSIS ENTRIES:", len(analysis_data))

print("\nENTRIES BY SOURCE:")
print(analysis_data["source"].value_counts())

print("\nLONGEST ENTRY:",
      analysis_data["word_count"].max())

print("\nENTRIES OVER 100 WORDS:",
      (analysis_data["word_count"] > 100).sum())

## 3. Check how well the sentiment model works

Before relying on the model, I manually labeled 40 feedback entries and compared my labels with the predictions. The baseline matched **70%** of those examples, with a macro F1 score around **0.70** and negative recall of **63.2%**.

This was a small evaluation sample, so I don't treat the score as proof the model is accurate on every review. One thing I noticed was that the model missed some negative comments when employees described difficult conditions without using obviously negative language.

The completed evaluation file is expected at `data/processed/sentiment_evaluation.csv` when running the notebook locally.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

evaluation = pd.read_csv("data/processed/sentiment_evaluation.csv")
for col in ["human_sentiment", "predicted_sentiment"]:
    evaluation[col] = evaluation[col].str.strip().str.lower()
labels = ["negative", "neutral", "positive"]
print(f"Baseline accuracy: {accuracy_score(evaluation.human_sentiment, evaluation.predicted_sentiment):.1%}")
print(classification_report(evaluation.human_sentiment, evaluation.predicted_sentiment, labels=labels, zero_division=0))
print(pd.DataFrame(confusion_matrix(evaluation.human_sentiment, evaluation.predicted_sentiment, labels=labels),index=labels,columns=labels))

## 4. Predict sentiment with RoBERTa

I used a pretrained RoBERTa model to give each feedback entry an estimated positive, neutral, or negative label. I saved the model's confidence score alongside the label so I could inspect its decisions later.

This is an overall passage sentiment prediction, not a confirmed opinion about every workplace issue mentioned in the passage. The first run downloads the model and requires the libraries listed in `requirements.txt`.

In [ ]:
from transformers import pipeline
sentiment_model = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=-1
)

In [ ]:
# step 25: Run sentiment analysis on all feedback

import pandas as pd

# Make a copy to preserve our prepared data
sentiment_data = analysis_data.copy()

# Run the existing RoBERTa model
predictions = sentiment_model(
    sentiment_data["feedback_text"].fillna("").tolist(),
    truncation=True,
    batch_size=8
)

# Store predicted sentiment
sentiment_data["sentiment"] = [
    result["label"] for result in predictions
]

# Store model confidence
sentiment_data["confidence"] = [
    round(result["score"], 3) for result in predictions
]

# Review sentiment distribution
print("SENTIMENT DISTRIBUTION:")
print(sentiment_data["sentiment"].value_counts())

print("\nSENTIMENT BY SOURCE:")
print(pd.crosstab(
    sentiment_data["source"],
    sentiment_data["sentiment"]
))

print("\nDATASET SHAPE:")
print(sentiment_data.shape)

display(sentiment_data[[
    "analysis_id",
    "source",
    "feedback_text",
    "sentiment",
    "confidence"
]].head(10))

## 5. Identify the workplace topics employees bring up

I tested a zero-shot theme model first, but I saw unrelated topics getting high scores. For my final exploratory analysis, I used keyword rules I could check and explain more easily.

The rules look for topics such as workload, scheduling, management, compensation, safety, and training. A match means the topic appears in the feedback. It doesn't automatically mean the employee is complaining about that topic. Some indirect mentions will be missed.

In [ ]:
# step 27: Create workplace theme detection rules

import re

theme_keywords = {
    "Management": [
        "manager", "supervisor", "leadership",
        "favoritism", "management"
    ],
    "Workload": [
        "workload", "exhausting", "physically",
        "heavy lifting", "labor intensive",
        "repetitive", "quota", "targets"
    ],
    "Compensation": [
        "pay", "salary", "wages", "benefits",
        "compensation", "bonus"
    ],
    "Scheduling": [
        "schedule", "shifts", "overtime",
        "hours", "breaks", "weekends"
    ],
    "Training": [
        "training", "onboarding", "orientation",
        "unprepared", "learning"
    ],
    "Career Growth": [
        "promotion", "advancement", "career",
        "growth opportunities"
    ],
    "Safety": [
        "safety", "injury", "hazard", "unsafe"
    ],
    "Coworkers": [
        "coworkers", "colleagues", "teamwork",
        "teammates"
    ]
}

def detect_themes(text):
    text = str(text).lower()
    found = []

    for theme, keywords in theme_keywords.items():
        for keyword in keywords:
            pattern = r"\b" + re.escape(keyword) + r"\b"

            if re.search(pattern, text):
                found.append(theme)
                break

    return found

# Test on the same 10 feedback entries
test_sample = sentiment_data.head(10).copy()

test_sample["detected_themes"] = (
    test_sample["feedback_text"].apply(detect_themes)
)

pd.set_option("display.max_colwidth", None)

display(test_sample[[
    "feedback_text",
    "detected_themes"
]])

pd.reset_option("display.max_colwidth")

In [ ]:
# step 28: Refine theme rules and classify feedback

# Expand existing theme keywords
theme_keywords["Career Growth"].extend([
    "grow", "career path", "promote", "moving up"
])

theme_keywords["Coworkers"].extend([
    "meet new people", "coworker", "team members",
    "colleagues"
])

theme_keywords["Management"].extend([
    "write up", "written up", "discipline"
])

# Add a new theme for employment stability
theme_keywords["Job Security"] = [
    "job security", "permanent", "temporary",
    "layoff", "fired", "termination",
    "contract renewal"
]

# Apply theme detection to the full dataset
theme_data = sentiment_data.copy()

theme_data["detected_themes"] = (
    theme_data["feedback_text"].apply(detect_themes)
)

# Count entries with at least one theme
theme_data["has_theme"] = (
    theme_data["detected_themes"].str.len() > 0
)

print("TOTAL ENTRIES:", len(theme_data))

print("ENTRIES WITH THEMES:",
      theme_data["has_theme"].sum())

print("ENTRIES WITHOUT THEMES:",
      (~theme_data["has_theme"]).sum())

# Count theme mentions
from collections import Counter

theme_counts = Counter(
    theme
    for themes in theme_data["detected_themes"]
    for theme in themes
)

print("\nTHEME FREQUENCIES:")
for theme, count in theme_counts.most_common():
    print(f"{theme}: {count}")

In [ ]:
# step 29: Analyze sentiment by workplace theme

# Keep entries with detected themes
themed_data = theme_data[
    theme_data["has_theme"]
].copy()

# Create one row per detected theme
themed_data = themed_data.explode(
    "detected_themes"
)

# Count sentiment predictions by theme
theme_sentiment = pd.crosstab(
    themed_data["detected_themes"],
    themed_data["sentiment"]
)

# Include all sentiment columns
theme_sentiment = theme_sentiment.reindex(
    columns=["negative", "neutral", "positive"],
    fill_value=0
)

# Calculate totals and negative percentage
theme_sentiment["total"] = (
    theme_sentiment[[
        "negative", "neutral", "positive"
    ]].sum(axis=1)
)

theme_sentiment["negative_pct"] = (
    theme_sentiment["negative"]
    / theme_sentiment["total"] * 100
).round(1)

# Sort by negative mention count
theme_sentiment = theme_sentiment.sort_values(
    "negative", ascending=False
)

display(theme_sentiment)

In [ ]:
# step 30: Aggregate themes by original record

# Start from the exploded theme data
record_themes = themed_data.copy()

# Create indicators for sentiment
record_themes["is_negative"] = (
    record_themes["sentiment"] == "negative"
).astype(int)

record_themes["is_positive"] = (
    record_themes["sentiment"] == "positive"
).astype(int)

record_themes["is_neutral"] = (
    record_themes["sentiment"] == "neutral"
).astype(int)

# One row per original record and theme
record_summary = (
    record_themes.groupby(
        ["record_id", "source", "detected_themes"],
        as_index=False
    )
    .agg(
        has_negative=("is_negative", "max"),
        has_neutral=("is_neutral", "max"),
        has_positive=("is_positive", "max")
    )
)

# Summarize original records by theme
final_theme_summary = (
    record_summary.groupby("detected_themes")
    .agg(
        records_mentioning_theme=("record_id", "nunique"),
        records_with_negative=("has_negative", "sum"),
        records_with_neutral=("has_neutral", "sum"),
        records_with_positive=("has_positive", "sum")
    )
)

# Calculate negative record percentage
final_theme_summary["negative_pct"] = (
    final_theme_summary["records_with_negative"]
    / final_theme_summary["records_mentioning_theme"]
    * 100
).round(1)

# Sort by number of records containing negative feedback
final_theme_summary = final_theme_summary.sort_values(
    "records_with_negative",
    ascending=False
)

display(final_theme_summary)

In [ ]:
# step 31: Compare workplace themes by data source

source_summary = (
    record_summary.groupby(
        ["source", "detected_themes"]
    )
    .agg(
        records_mentioning_theme=("record_id", "nunique"),
        records_with_negative=("has_negative", "sum")
    )
    .reset_index()
)

# Calculate percentage
source_summary["negative_pct"] = (
    source_summary["records_with_negative"]
    / source_summary["records_mentioning_theme"]
    * 100
).round(1)

# Sort results
source_summary = source_summary.sort_values(
    ["source", "records_with_negative"],
    ascending=[True, False]
)

print("THEME COMPARISON BY SOURCE:")
display(source_summary)

In [ ]:
# step 32: Review key negative feedback examples

priority_themes = [
    "Workload",
    "Management",
    "Scheduling"
]

# Filter negative passages for priority themes
quality_check = themed_data[
    (themed_data["detected_themes"].isin(priority_themes)) &
    (themed_data["sentiment"] == "negative")
].copy()

# Sort by model confidence
quality_check = quality_check.sort_values(
    "confidence",
    ascending=False
)

# Keep one example per original record and theme
quality_check = quality_check.drop_duplicates(
    subset=["record_id", "detected_themes"]
)

# Show up to five examples per theme
quality_sample = (
    quality_check.groupby("detected_themes")
    .head(5)
)

pd.set_option("display.max_colwidth", None)

display(quality_sample[[
    "record_id",
    "source",
    "detected_themes",
    "feedback_text",
    "sentiment",
    "confidence"
]])

pd.reset_option("display.max_colwidth")

## 6. Look at examples behind the numbers

I pulled examples connected to the main negative themes so I could check whether the labels made sense. This matters because the model sometimes labels an entire passage negative even if only one part of the passage is a complaint.

These examples are for a quick quality check, not a full validation of every theme label.

In [ ]:
# step 32: Review key negative feedback examples

priority_themes = [
    "Workload",
    "Management",
    "Scheduling"
]

# Filter negative passages for priority themes
quality_check = themed_data[
    (themed_data["detected_themes"].isin(priority_themes)) &
    (themed_data["sentiment"] == "negative")
].copy()

# Sort by model confidence
quality_check = quality_check.sort_values(
    "confidence",
    ascending=False
)

# Keep one example per original record and theme
quality_check = quality_check.drop_duplicates(
    subset=["record_id", "detected_themes"]
)

# Show up to five examples per theme
quality_sample = (
    quality_check.groupby("detected_themes")
    .head(5)
)

pd.set_option("display.max_colwidth", None)

display(quality_sample[[
    "record_id",
    "source",
    "detected_themes",
    "feedback_text",
    "sentiment",
    "confidence"
]])

pd.reset_option("display.max_colwidth")

## 7. Compare the findings and make charts

I created two charts: one showing how many original records contained negative, theme-matched passages, and another showing the share of theme-mentioning records with at least one negative passage.

I also compared Glassdoor with YouTube and looked at regular versus part-time Glassdoor reviews. The smaller groups have limited data, so I look at the record counts along with the percentages.

In [ ]:
# step 33: Visualize negative workplace feedback

import matplotlib.pyplot as plt

# Prepare chart data
chart_data = final_theme_summary.sort_values(
    "records_with_negative",
    ascending=True
)

# Create chart
plt.figure(figsize=(11, 6))

bars = plt.barh(
    chart_data.index,
    chart_data["records_with_negative"],
    color="#C94C4C"
)

# Add labels
for bar in bars:
    width = bar.get_width()

    plt.text(
        width + 0.3,
        bar.get_y() + bar.get_height() / 2,
        str(int(width)),
        va="center"
    )

plt.title(
    "Amazon Fulfillment Center: Negative Feedback by Workplace Theme",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Original Records Containing Negative Theme-Matched Feedback")
plt.ylabel("Workplace Theme")

plt.xlim(
    0,
    chart_data["records_with_negative"].max() + 5
)

plt.tight_layout()

# Save chart
plt.savefig(
    "data/processed/negative_feedback_by_theme.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# step 34: Visualize negative sentiment rates by theme

import matplotlib.pyplot as plt

# Sort workplace themes by negative percentage
rate_data = final_theme_summary.sort_values(
    "negative_pct",
    ascending=True
)

plt.figure(figsize=(11, 6))

bars = plt.barh(
    rate_data.index,
    rate_data["negative_pct"],
    color="#3978A8"
)

# Add percentage labels
for bar in bars:
    plt.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.1f}%",
        va="center"
    )

plt.title(
    "Negative Sentiment Rate by Workplace Theme",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "Percentage of Theme-Mentioning Records With Negative Passages"
)
plt.ylabel("Workplace Theme")
plt.xlim(0, 100)

plt.tight_layout()

plt.savefig(
    "data/processed/negative_sentiment_rate.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# step 35: Compare employment groups

# Select Glassdoor records
employment_data = record_summary[
    record_summary["source"] == "Glassdoor"
].copy()

# Attach employee status from original reviews
employee_info = (
    combined[combined["source"] == "Glassdoor"][
        ["record_id", "employee_status"]
    ]
    .drop_duplicates("record_id")
)

employment_data = employment_data.merge(
    employee_info,
    on="record_id",
    how="left"
)

# Focus on the two largest employment groups
employment_data = employment_data[
    employment_data["employee_status"].isin(
        ["REGULAR", "PART_TIME"]
    )
]

# Summarize negative theme-matched records
employment_summary = (
    employment_data.groupby(
        ["employee_status", "detected_themes"]
    )
    .agg(
        records_mentioning_theme=("record_id", "nunique"),
        records_with_negative=("has_negative", "sum")
    )
    .reset_index()
)

# Calculate percentages
employment_summary["negative_pct"] = (
    employment_summary["records_with_negative"]
    / employment_summary["records_mentioning_theme"]
    * 100
).round(1)

display(employment_summary.sort_values(
    ["employee_status", "records_with_negative"],
    ascending=[True, False]
))

## 8. Save the results and business recommendations

Once I finished the analysis, I exported the summary tables and charts so I could use them in my GitHub portfolio. I also linked the main findings to ideas Amazon could test, such as task rotation, more predictable schedules, and better manager communication.

These are recommendations to investigate, not proven ways to reduce turnover.

In [ ]:
# step 36: Save analysis results

import os

os.makedirs("data/processed", exist_ok=True)

# Save passage-level predictions
theme_data.to_csv(
    "data/processed/feedback_with_sentiment_themes.csv",
    index=False
)

# Save original-record-level analysis
record_summary.to_csv(
    "data/processed/record_theme_summary.csv",
    index=False
)

# Save summary tables
final_theme_summary.to_csv(
    "data/processed/final_theme_summary.csv"
)

source_summary.to_csv(
    "data/processed/source_comparison.csv",
    index=False
)

employment_summary.to_csv(
    "data/processed/employment_comparison.csv",
    index=False
)

print("Analysis files saved successfully!")

In [ ]:
# step 37: Create business recommendations

recommendations = pd.DataFrame([
    {
        "theme": "Workload",
        "finding": "Physical exhaustion and repetitive tasks",
        "recommendation": "Introduce task rotation and workload reviews",
        "success_metric": "Fatigue reports, injury rates, 90-day retention"
    },
    {
        "theme": "Scheduling",
        "finding": "Long shifts, overtime, and restricted breaks",
        "recommendation": "Improve shift planning and review overtime practices",
        "success_metric": "Schedule satisfaction and overtime hours"
    },
    {
        "theme": "Management",
        "finding": "Poor communication and productivity pressure",
        "recommendation": "Introduce manager coaching and regular employee check-ins",
        "success_metric": "Engagement scores and management complaints"
    }
])

display(recommendations)

recommendations.to_csv(
    "data/processed/business_recommendations.csv",
    index=False
)

print("Business recommendations saved successfully!")

## What I took away from the analysis

Workload, scheduling, and management showed up as key topics alongside negative feedback. My original project focused on training, but the updated results made me rethink which issues deserve attention first.

A few things matter when reading these results:

- **148 source records, not 739 employees:** The analysis includes 138 Glassdoor reviews and 10 videos. Splitting the feedback created 739 analysis entries, including overlapping YouTube passages.
- **One record is counted once per theme:** This helps keep long videos from dominating the theme summaries.
- **Sentiment isn't theme-specific:** A negative label applies to a whole passage and doesn't always mean every topic in the passage is criticized.
- **The model isn't perfect:** On my manually reviewed set, its accuracy was 70%, and it missed some negative comments. The keyword rules also have gaps.
- **No direct turnover measurement:** I didn't have employee exit records, so I can't claim these workplace concerns caused people to quit.

My biggest lesson was to check what a model gets wrong before turning predictions into business conclusions. This project helped me practice Python and NLP while thinking more critically about how to use analysis to support operational decisions.